# ai03 DIY Task Solutions

**INSTRUCTOR SOLUTIONS - DO NOT DISTRIBUTE**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print("✓ Libraries imported")

In [ ]:
heartData = pd.read_csv('heart_disease_uci.csv')
print(f"✓ Dataset loaded: {heartData.shape}")

In [ ]:
print(heartData.head())

In [ ]:
print(heartData.info())

In [ ]:
print(heartData.describe())

In [ ]:
print("Missing values:")
print(heartData.isnull().sum())

In [ ]:
heartDataClean = heartData.dropna()
print(f"✓ Cleaned data shape: {heartDataClean.shape}")

In [ ]:
print("Data types:")
print(heartDataClean.dtypes)

In [ ]:
# One-hot encode categorical columns
heartDataClean = pd.get_dummies(heartDataClean, drop_first=True)

print(f"✓ Final cleaned shape: {heartDataClean.shape}")
print(f"Columns: {list(heartDataClean.columns)}")

In [ ]:
# Separate features and target
# Note: assuming 'target' is the name (may vary by dataset)
targetColumn = 'target'
if targetColumn not in heartDataClean.columns:
    # Try to find the target column
    possible_targets = [col for col in heartDataClean.columns if 'target' in col.lower() or 'disease' in col.lower()]
    if possible_targets:
        targetColumn = possible_targets[0]

X = heartDataClean.drop(targetColumn, axis=1)
y = heartDataClean[targetColumn]

print(f"✓ Features shape: {X.shape}")
print(f"✓ Target shape: {y.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

In [ ]:
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✓ Training set: {xTrain.shape[0]} rows")
print(f"✓ Testing set: {xTest.shape[0]} rows")

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(xTrain, yTrain)

print("✓ Model trained!")

In [ ]:
predictions = model.predict(xTest)

print(f"✓ Predictions made for {len(predictions)} patients")

In [ ]:
accuracy = accuracy_score(yTest, predictions)

print(f"\n{'='*50}")
print(f"ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}")

In [ ]:
print(classification_report(yTest, predictions, target_names=['No Disease', 'Disease']))

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=X.columns, class_names=['No Disease', 'Disease'],
          filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree for Heart Disease Prediction")
plt.tight_layout()
plt.show()

In [ ]:
importances = model.feature_importances_

featureImportance = list(zip(X.columns, importances))
featureImportance.sort(key=lambda x: x[1], reverse=True)

print("\nTop 5 Most Important Features:")
print("="*40)
for i, (feature, importance) in enumerate(featureImportance[:5], 1):
    print(f"{i}. {feature:20} {importance:.4f}")

In [ ]:
print("Testing different max_depth values...\n")
print("max_depth | Train Accuracy | Test Accuracy")
print("-" * 45)

for depth in range(2, 9):
    tempModel = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tempModel.fit(xTrain, yTrain)
    
    trainPred = tempModel.predict(xTrain)
    testPred = tempModel.predict(xTest)
    
    trainAcc = accuracy_score(yTrain, trainPred)
    testAcc = accuracy_score(yTest, testPred)
    
    print(f"    {depth}    |    {trainAcc:.4f}     |   {testAcc:.4f}")

In [ ]:
# Compare different max_depth values
max_depths = range(2, 9)
trainAccs = []
testAccs = []

for depth in max_depths:
    tempModel = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tempModel.fit(xTrain, yTrain)
    trainAccs.append(accuracy_score(yTrain, tempModel.predict(xTrain)))
    testAccs.append(accuracy_score(yTest, tempModel.predict(xTest)))

plt.figure(figsize=(10, 6))
plt.plot(max_depths, trainAccs, marker='o', label='Training Accuracy')
plt.plot(max_depths, testAccs, marker='s', label='Test Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Heart Disease Model: Max Depth vs Accuracy')
plt.legend()
plt.grid(True)
plt.show()